In [5]:
import pandas as pd
import numpy as np


df = pd.read_csv('/content/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [6]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

Task 1: Regional comparison bar chart

In [7]:
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [14]:
import plotly.express as px

fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    title='Western Europe is the happiest region, revealing a clear gap in global well-being',
    labels={'Happiness_Score': 'Average Happiness Score', 'Region': ''},
    color='Happiness_Score',
    color_continuous_scale='Sunset'
)


fig.update_layout(
    xaxis=dict(range=[0, region_avg['Happiness_Score'].max() + 1]),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis_gridcolor='#EEEEEE',
    yaxis_gridcolor='white'

)

fig.update_traces(marker_line_width=0)

fig.show()

Task 2: Top 8 vs. Bottom 8 contrast

In [15]:
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")


Global average: 5.81


In [24]:
import plotly.express as px
import pandas as pd

# Get top 8 and bottom 8
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

# Combine and sort
combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')

# Global average
global_avg = df['Happiness_Score'].mean()

# Create chart
fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={
        'Top 8': '#52B788',     # gold (happy)
        'Bottom 8': '#5E81AC'   # blue (less happy)
    },
    title='A dramatic global gap: the happiest countries score nearly double the least happy',
    labels={'Happiness_Score': 'Happiness Score', 'Country': ''}
)

# Add global average line (visual separator)
fig.add_vline(
    x=global_avg,
    line_dash="dash",
    line_color="black"
)

# Add annotation
fig.add_annotation(
    x=global_avg,
    y=0,
    text="Global Average",
    showarrow=False,
    yshift=10
)

# Clean layout + zero baseline
fig.update_layout(
    xaxis=dict(range=[0, combined['Happiness_Score'].max() + 1]),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis_gridcolor='#EEEEEE',
    yaxis_gridcolor='white'
)

# Final polish
fig.update_traces(marker_line_width=0)

fig.show()

Stretch Goal

In [30]:
import plotly.express as px
import pandas as pd

# Select required regions
regions = [
    'Western Europe',
    'Latin America and Caribbean',
    'East Asia',
    'Sub-Saharan Africa',
    'South Asia'
]

filtered = df[df['Region'].isin(regions)]

# Compute averages
region_factors = (
    filtered.groupby('Region')[['GDP', 'Freedom']]
    .mean()
    .reset_index()
)

# Convert to long format
region_long = region_factors.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Factor',
    value_name='Value'
)

# Create grouped bar chart
fig = px.bar(
    region_long,
    x='Region',
    y='Value',
    color='Factor',
    barmode='group',
    title='Western Europe leads in both economic strength and freedom, highlighting global inequality across regions',
    labels={'Value': 'Average Score', 'Region': '', 'Factor': ''},
    color_discrete_map={
        'GDP': '#52B788',       # pleasant green
        'Freedom': '#5E81AC'    # soft blue
    }
)

# Clean layout
fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis_gridcolor='#EEEEEE',
    yaxis_gridcolor='#EEEEEE',
    legend_title_text=''
)

# Final polish
fig.update_traces(marker_line_width=0)

fig.show()